In [ ]:
1. What is HITL?
Human-in-the-Loop is a design pattern where a human actively participates at critical points in an AI workflow to
supervise, approve, correct, or guide the model's output 
. It acts as a "human checkpoint" within an AI pipeline to ensure high-stakes decisions aren't made autonomously.

2. Why do we need HITL?
    Lack of Perfection: Current LLMs can misinterpret goals, struggle with ambiguity (e.g., "next Friday"), or hallucinate 

    Accountability: AI cannot be "blamed" for mistakes. For financial transactions or sensitive emails, a human must take responsibility 

    Safety: Prevents destructive actions (e.g., deleting critical files) by requiring confirmation 

    Ethical Alignment: Humans can add empathy and align responses with company core values that an LLM might miss 


3. Common HITL Patterns
    Action Approval: The most common pattern. A human must approve sensitive actions like payments or file deletions 

    Output Review/Edit: A human reviews and refines drafts (e.g., a blog post) before they are published 

    Ambiguity Clarification: The agent pauses to ask the user for more details when a query is unclear 

    Escalation: When an agent cannot handle a case, it transfers the "state" to a human representative 


4. Technical Implementation in LangGraph
LangGraph implements HITL using two primary mechanisms:

interrupt() Function: This pauses the graph's execution, saves the current state using a Checkpointer (like MemorySaver), and sends a message to the front end 

Command Object: Once the human provides input (Yes/No/Edit), the front end re-invokes the graph using Command(resume=...).
The graph then loads the state from the checkpointer and continues exactly where it left off 

*****************************************************************************************************************************************************
Interview Questions
Conceptual Questions
Explain the "Action Approval" pattern in Agentic AI. Why is it critical for enterprise applications?

Answer Hint: Focus on risk mitigation, financial safety, and the "human-in-the-loop" acting as a gatekeeper for irreversible actions.

How does HITL improve the "Accountability" of an AI agent?

Answer Hint: Discuss legal and ethical responsibility. Since an AI cannot be held liable, a human must sign off on decisions to ensure a person is responsible for the outcome.

What is the difference between an autonomous agent and a HITL-enabled agent?

Answer Hint: Autonomous agents complete the entire loop (Start → Action → End) without intervention, while HITL agents include external breakpoints for human judgment.

Technical (LangGraph Specific) Questions
How does LangGraph maintain the "state" of an agent when it is interrupted for human input?

Answer Hint: Mention Checkpointers and Thread IDs. The state is serialized and saved in memory or a database so the execution can be resumed later 


Describe the role of the interrupt and Command functions in a LangGraph workflow.

Answer Hint: interrupt signals the pause and packages data for the UI; Command is used by the UI to send the human's decision back to the graph to resume processing 


Why is a thread_id necessary when implementing HITL in LangGraph?

Answer Hint: The thread_id allows the checkpointer to uniquely identify which conversation state to load when the human provides their input 


In [ ]:
from typing import Annotated

from langchain_openai import ChatOpenAI
from langchain_core.messages import AnyMessage, AIMessage

from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import interrupt, Command
from dotenv import load_dotenv
from typing import TypedDict, Annotated
from langchain_core.messages import BaseMessage

In [ ]:
load_dotenv()

In [ ]:

llm = ChatOpenAI(model="gpt-4.1-mini")

In [ ]:
from langgraph.graph.message import add_messages

class ChatState(TypedDict):

    messages: Annotated[list[BaseMessage], add_messages]

In [ ]:
def chat_node(state: ChatState):

    decision = interrupt({
        "type": "approval",
        "reason": "Model is about to answer a user question.",
        "question": state["messages"][-1].content,
        "instruction": "Approve this question? yes/no"
    })
    
    if decision["approved"] == 'no':
        return {"messages": [AIMessage(content="Not approved.")]}

    else:
        response = llm.invoke(state["messages"])
        return {"messages": [response]}

In [ ]:
# 3. Build the graph: START -> chat -> END
builder = StateGraph(ChatState)

builder.add_node("chat", chat_node)

builder.add_edge(START, "chat")
builder.add_edge("chat", END)

# Checkpointer is required for interrupts
checkpointer = MemorySaver()

# Compile the app
app = builder.compile(checkpointer=checkpointer)

In [ ]:
# Create a new thread id for this conversation
config = {"configurable": {"thread_id": '1234'}}

# ---- STEP 1: user asks a question ----
initial_input = {
    "messages": [
        ("user", "Explain gradient descent in very simple terms.")
    ]
}

# Invoke the graph for the first time
result = app.invoke(initial_input, config=config)

In [ ]:
result

In [ ]:
message = result['__interrupt__'][0].value
message

In [ ]:
user_input = input(f"\nBackend message - {message} \n Approve this question? (y/n): ")

In [ ]:
# Resume the graph with the approval decision
final_result = app.invoke(
    Command(resume={"approved": user_input}),
    config=config,
)

In [ ]:
print(final_result["messages"][-1].content)

In [ ]:
# backend.py

from langgraph.graph import StateGraph, START
from typing import TypedDict, Annotated
from langchain_core.messages import BaseMessage, HumanMessage
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_core.tools import tool
from langgraph.types import interrupt, Command
from dotenv import load_dotenv
import requests

load_dotenv()

# -------------------
# 1. LLM
# -------------------
llm = ChatOpenAI()

# -------------------
# 2. Tools
# -------------------
@tool
def get_stock_price(symbol: str) -> dict:
    """
    Fetch latest stock price for a given symbol (e.g. 'AAPL', 'TSLA') 
    using Alpha Vantage with API key in the URL.
    """
    url = (
        "https://www.alphavantage.co/query"
        f"?function=GLOBAL_QUOTE&symbol={symbol}&apikey=C9PE94QUEW9VWGFM"
    )
    r = requests.get(url)
    return r.json()


@tool
def purchase_stock(symbol: str, quantity: int) -> dict:
    """
    Simulate purchasing a given quantity of a stock symbol.

    HUMAN-IN-THE-LOOP:
    Before confirming the purchase, this tool will interrupt
    and wait for a human decision ("yes" / anything else).
    """
    # This pauses the graph and returns control to the caller
    decision = interrupt(f"Approve buying {quantity} shares of {symbol}? (yes/no)")

    if isinstance(decision, str) and decision.lower() == "yes":
        return {
            "status": "success",
            "message": f"Purchase order placed for {quantity} shares of {symbol}.",
            "symbol": symbol,
            "quantity": quantity,
        }
    
    else:
        return {
            "status": "cancelled",
            "message": f"Purchase of {quantity} shares of {symbol} was declined by human.",
            "symbol": symbol,
            "quantity": quantity,
        }


tools = [get_stock_price, purchase_stock]
llm_with_tools = llm.bind_tools(tools)

# -------------------
# 3. State
# -------------------
class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

# -------------------
# 4. Nodes
# -------------------
def chat_node(state: ChatState):
    """LLM node that may answer or request a tool call."""
    messages = state["messages"]
    response = llm_with_tools.invoke(messages)
    return {"messages": [response]}

tool_node = ToolNode(tools)

# -------------------
# 5. Checkpointer (in-memory)
# -------------------
memory = MemorySaver()

# -------------------
# 6. Graph
# -------------------
graph = StateGraph(ChatState)
graph.add_node("chat_node", chat_node)
graph.add_node("tools", tool_node)

graph.add_edge(START, "chat_node")

graph.add_conditional_edges("chat_node", tools_condition)
graph.add_edge("tools", "chat_node")

chatbot = graph.compile(checkpointer=memory)

# -------------------
# 7. Simple usage example (CLI with HITL)
# -------------------
if __name__ == "__main__":
    
    # Use a fixed thread_id so the conversation is persisted in memory
    thread_id = "demo-thread"

    while True:
        user_input = input("You: ")
        if user_input.lower().strip() in {"exit", "quit"}:
            print("Goodbye!")
            break

        # Build initial state for this turn
        state = {"messages": [HumanMessage(content=user_input)]}

        # Run the graph (may hit an interrupt)
        result = chatbot.invoke(
            state,
            config={"configurable": {"thread_id": thread_id}},
        )

        # Check for HITL interrupt from purchase_stock
        interrupts = result.get("__interrupt__", [])

        if interrupts:
            # Our interrupt payload is the string we passed to interrupt(...)
            prompt_to_human = interrupts[0].value
            print(f"HITL: {prompt_to_human}")
            decision = input("Your decision: ").strip().lower()

            # Resume graph with the human decision ("yes" / "no" / whatever)
            result = chatbot.invoke(
                Command(resume=decision),
                config={"configurable": {"thread_id": thread_id}},
            )

        # Get the latest message from the assistant
        messages = result["messages"]
        last_msg = messages[-1]
        print(f"Bot: {last_msg.content}\n")